# 01 — Explore the FPL API

Goal: get comfortable with what the free API returns before trusting the pipeline. Everything here hits the network directly and is throwaway; nothing is saved.

Reference: `docs/FPL_API_REFERENCE.md`.

In [ ]:
# Run this cell first in every notebook.
# It makes the package importable from the repo root and loads settings from config/settings.toml.
# autoreload: edits under src/ take effect on the next cell run, no kernel restart needed.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from fpl_analysis.config import load_settings
from fpl_analysis.store import connect, query

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
settings = load_settings()
print("project root:", settings.project_root)
print("warehouse   :", settings.duckdb_path, "| exists:", settings.duckdb_path.exists())

In [ ]:
from fpl_analysis.api import FPLClient, current_and_next_gameweek

client = FPLClient(base_url=settings.base_url)
bootstrap = client.bootstrap_static()
print("top-level keys:", list(bootstrap))
print("players:", len(bootstrap["elements"]), "| teams:", len(bootstrap["teams"]), "| gameweeks:", len(bootstrap["events"]))
print("current / next GW:", current_and_next_gameweek(bootstrap))

## Players

Note how many numeric fields arrive as **strings** (`form`, `ict_index`, `expected_goals`...). That is why `sql/staging/040_stg_players.sql` casts everything once.

In [ ]:
players = pd.DataFrame(bootstrap["elements"])
teams = pd.DataFrame(bootstrap["teams"]).set_index("id")
positions = {p["id"]: p["singular_name_short"] for p in bootstrap["element_types"]}
players["team_name"] = players["team"].map(teams["short_name"])
players["pos"] = players["element_type"].map(positions)
players["price"] = players["now_cost"] / 10
cols = ["web_name", "team_name", "pos", "price", "status", "total_points", "form", "selected_by_percent",
        "minutes", "expected_goal_involvements_per_90", "ep_next"]
players.sort_values("total_points", ascending=False)[cols].head(20)

In [ ]:
players.dtypes.to_frame("dtype").T

## Fixtures and FDR

`team_h_difficulty` is the difficulty **for the home team**, `team_a_difficulty` for the away team.

In [ ]:
fixtures = pd.DataFrame(client.fixtures())
fixtures["home"] = fixtures["team_h"].map(teams["short_name"])
fixtures["away"] = fixtures["team_a"].map(teams["short_name"])
current_gw, next_gw = current_and_next_gameweek(bootstrap)
fixtures[fixtures["event"] == next_gw][["event", "kickoff_time", "home", "away", "team_h_difficulty", "team_a_difficulty"]]

## Your squad (entry endpoints are public — no login needed)

In [ ]:
entry = client.entry(settings.entry_id)
print(entry["name"], "-", entry["player_first_name"], entry["player_last_name"])
print("overall points:", entry["summary_overall_points"], "| rank:", f'{entry["summary_overall_rank"]:,}')
picks = client.entry_picks(settings.entry_id, current_gw)
squad = pd.DataFrame(picks["picks"]).merge(players[["id", "web_name", "team_name", "pos", "price"]], left_on="element", right_on="id")
print("bank:", picks["entry_history"]["bank"] / 10, "| value:", picks["entry_history"]["value"] / 10, "| chip:", picks["active_chip"])
squad[["position", "web_name", "team_name", "pos", "price", "is_captain", "is_vice_captain", "multiplier"]]

## One player's per-gameweek history

This is the expensive endpoint (one call per player). The pipeline fetches it for everyone in `fpl refresh`; skip it with `--no-history` on a quick run.

In [ ]:
top_id = int(players.sort_values("total_points", ascending=False)["id"].iloc[0])
summary = client.element_summary(top_id)
pd.DataFrame(summary["history"])[["round", "opponent_team", "was_home", "minutes", "total_points", "expected_goals", "expected_assists", "bonus", "value"]]